In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [3]:
# Try (on Groq):

# llama3-70b-8192 ✅ (better reasoning)
# mixtral-8x7b ✅ (better tool usage)
# llama-3.1-8b-instant
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)

In [4]:
from langchain.tools import tool 

@tool
def get_weather(location:str)->str:
    """get the weather of the city"""
    return f"the weather of {location} is sunny"

model_with_tools = model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("whats the weather like in delhi?")
response

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'c5afnd5ew', 'function': {'arguments': '{"location":"delhi"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 221, 'total_tokens': 236, 'completion_time': 0.032289186, 'completion_tokens_details': None, 'prompt_time': 0.013547622, 'prompt_tokens_details': None, 'queue_time': 0.063701798, 'total_time': 0.045836808}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d28a1-ddd9-77c0-8c29-ba1b51ef307b-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'delhi'}, 'id': 'c5afnd5ew', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 15, 'total_tokens': 236})

In [6]:
for tool_call in response.tool_calls:
    print(f"tool: {tool_call['name']}")
    print(f"args: {tool_call['args']}")

tool: get_weather
args: {'location': 'delhi'}


TOOL EXECUTION-MANUAL PROCESS

In [7]:
#Step-1 - MODEL GENERATES TOOL CALLS
messages = [{"role":"user","content":"whats the weather like in boston?"}]

response = model_with_tools.invoke(messages)
messages.append(response)
# print(response)

#Step-2 - execute tools and collect results
for tool_call in response.tool_calls:
   tool_result = get_weather.invoke(tool_call)
   messages.append(tool_result)
 
# print(tool_result)
print("messages=>",messages)

#Step-3 - Pass results back to model for final response
result = model_with_tools.invoke(messages)
result


messages=> [{'role': 'user', 'content': 'whats the weather like in boston?'}, AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bzzhpyrsr', 'function': {'arguments': '{"location":"boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 221, 'total_tokens': 236, 'completion_time': 0.02818671, 'completion_tokens_details': None, 'prompt_time': 0.016030776, 'prompt_tokens_details': None, 'queue_time': 0.047024924, 'total_time': 0.044217486}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d28a1-e22a-7380-bad6-1ac26533d78d-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'boston'}, 'id': 'bzzhpyrsr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 15, 'total_tokens': 236}), ToolMessage(cont

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'rg4b69tz1', 'function': {'arguments': '{"location":"boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 253, 'total_tokens': 305, 'completion_time': 0.096043998, 'completion_tokens_details': None, 'prompt_time': 0.018211316, 'prompt_tokens_details': None, 'queue_time': 0.046987003, 'total_time': 0.114255314}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d28a1-e2b3-7531-acc8-2a469d885a5a-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'boston'}, 'id': 'rg4b69tz1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 253, 'output_tokens': 52, 'total_tokens': 305})

In [8]:
# if response.tool_calls:
#     for tool_call in response.tool_calls:
#         tool_result = get_weather.invoke(tool_call)
#         messages.append(tool_result)
# else:
#     print("❌ Model did not call tool")

In [9]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Use this tool to get the current weather for a given location"""
    return f"The weather in {location} is sunny"

llm = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt = 
    """
    You are a helpful assistant.
    Do NOT invent tools.
    Use the tool whenever user asks about weather.
    """
)

response = agent.invoke({
    "messages": [("user", "What's the weather in Boston?")]
})

print(response)

{'messages': [HumanMessage(content="What's the weather in Boston?", additional_kwargs={}, response_metadata={}, id='0962cf65-a0c6-4a75-ab77-bc3704bff100'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '2xj709jnd', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 253, 'total_tokens': 267, 'completion_time': 0.026521536, 'completion_tokens_details': None, 'prompt_time': 0.077551125, 'prompt_tokens_details': None, 'queue_time': 0.131303234, 'total_time': 0.104072661}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d28a1-fc79-75b2-874a-132f96d1d180-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '2xj709jnd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'i

In [10]:
# What YOU did (manual loop)
# 1. model.invoke()
# 2. check tool_calls
# 3. run function manually
# 4. append result
# 5. call model again

# ✔ This is correct logic
# ❌ But manual and error-prone

In [11]:
# What Agents do (automatic)

# Using LangChain:

# Agent
#   ↳ calls LLM
#   ↳ detects tool call
#   ↳ executes tool
#   ↳ feeds result back
#   ↳ repeats until final answer

# 👉 You don’t write the loop — agent handles everything